# Pharmacy Sales – End-to-End ML Project
**الهدف:** توقع إيراد الشهر الجاي (LKR) لكل فرع × فئة دواء.

الخطوات: 1) Preprocessing  2) EDA  3) Feature Engineering & Modeling  4) Evaluation  5) Deployment

> الملف الخام في `../data/raw/` (شغّلي الـ notebook من فولدر `notebooks/`).

In [ ]:
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)
RAW = "../data/raw/Pharmacy_OLTP_SLStyle_18Months.csv"

## 1) Data Preprocessing

In [ ]:
df = pd.read_csv(RAW)
print(df.shape)
df.head()

In [ ]:
# أنواع البيانات
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], format="%m/%d/%Y")
df["Expiry_Date"] = pd.to_datetime(df["Expiry_Date"], format="%m/%d/%Y")
for c in ["Branch_Name","District","Province","Payment_Method","Customer_Gender","Customer_City",
          "Medicine_Category","Dosage_Form","Supplier_Name","Prescription_Required"]:
    df[c] = df[c].astype("category")

# مكرر وناقص
print("Duplicate rows:", df.duplicated().sum(), "| duplicate Transaction_ID:", df.Transaction_ID.duplicated().sum())
print(df.isna().sum()[df.isna().sum() > 0])
print(df.groupby("Dosage_Form", observed=True).Strength_mg.apply(lambda s: s.isna().mean()).round(2))

`Strength_mg` ناقصة 100% في Cream / Syrup / Ointment / Sachet / Inhaler لأن الـ mg مش منطقي لها، فبنحط 0 ونضيف flag.

In [ ]:
df["Has_Strength"] = df["Strength_mg"].notna().astype(int)
df["Strength_mg"] = df["Strength_mg"].fillna(0)

# فحوصات الاتساق
calc = (df.Quantity * df.Unit_Price_LKR * (1 - df.Discount_Rate)).round(2)
print("Line_Total mismatches:", (abs(calc - df.Line_Total_LKR) > 0.02).sum())
print("Sold after expiry:", (df.Expiry_Date < df.Transaction_Date).sum())
print("Non-positive qty/price:", ((df.Quantity <= 0) | (df.Unit_Price_LKR <= 0)).sum())

# Outliers: نعلّمها ومنحذفهاش (الفواتير الكبيرة مبيعات حقيقية)
q1, q3 = df.Line_Total_LKR.quantile([.25, .75]); upper = q3 + 1.5 * (q3 - q1)
df["Is_High_Value"] = (df.Line_Total_LKR > upper).astype(int)
print("High-value flagged:", df.Is_High_Value.sum())

# أعمدة مشتقة
df["Days_To_Expiry"] = (df.Expiry_Date - df.Transaction_Date).dt.days
df["YearMonth"] = df.Transaction_Date.dt.to_period("M").astype(str)
df["DayOfWeek"] = df.Transaction_Date.dt.dayofweek
df["Quarter"] = df.Transaction_Date.dt.quarter
df["Age_Group"] = pd.cut(df.Customer_Age, [17, 30, 45, 60, 80], labels=["18-30","31-45","46-60","61-80"])
df = df.sort_values("Transaction_Date").reset_index(drop=True)
REV = "Line_Total_LKR"
df.shape

## 2) Exploratory Data Analysis

In [ ]:
m = df.groupby("YearMonth").agg(Revenue=(REV, "sum"), Units=("Quantity", "sum"), Txns=("Transaction_ID", "count"))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
m.Revenue.plot(ax=ax[0], marker="o"); ax[0].set_title("Monthly revenue (LKR)")
m.Txns.plot(ax=ax[1], marker="o", color="tab:green"); ax[1].set_title("Monthly transactions")
for a in ax: a.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()
print("Monthly revenue CV:", round(m.Revenue.std() / m.Revenue.mean(), 3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df[REV], bins=50, ax=ax[0]); ax[0].set_title("Line total (LKR)")
sns.histplot(df.Customer_Age, bins=30, ax=ax[1]); ax[1].set_title("Customer age")
sns.countplot(x="Quantity", data=df, ax=ax[2]); ax[2].set_title("Quantity per line")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
df.groupby("Medicine_Category", observed=True)[REV].sum().sort_values().plot.barh(ax=ax[0]); ax[0].set_title("Revenue by category")
df.groupby("Branch_Name", observed=True)[REV].sum().sort_values().plot.barh(ax=ax[1], color="tab:orange"); ax[1].set_title("Revenue by branch")
df.groupby("Payment_Method", observed=True)[REV].sum().plot.pie(ax=ax[2], autopct="%1.0f%%"); ax[2].set_ylabel(""); ax[2].set_title("Revenue by payment")
plt.tight_layout(); plt.show()

In [ ]:
pv = df.pivot_table(index="Medicine_Category", columns="Branch_Name", values=REV, aggfunc="sum", observed=True)
plt.figure(figsize=(11, 7)); sns.heatmap(pv / 1e6, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Revenue (million LKR): category x branch"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.groupby("Month")[REV].sum().plot.bar(ax=ax[0]); ax[0].set_title("Revenue by calendar month (2022 = 12 mo, 2023 = 6 mo)")
df.groupby("DayOfWeek")[REV].sum().plot.bar(ax=ax[1], color="tab:purple"); ax[1].set_title("Revenue by day of week (0=Mon)")
plt.tight_layout(); plt.show()

num = ["Customer_Age","Pack_Size","Strength_mg","Quantity","Unit_Price_LKR","Discount_Rate","Days_To_Expiry",REV]
plt.figure(figsize=(8, 6)); sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0); plt.title("Correlation"); plt.show()

In [ ]:
print((df.groupby("Medicine_Category", observed=True)[REV].sum().sort_values(ascending=False) / 1e6).round(1).head(5))
print(df.groupby("Payment_Method", observed=True)[REV].agg(["count", "mean"]).round(0))
print(df.groupby("Customer_Gender", observed=True)[REV].agg(["sum", "mean"]).round(0))
print("Rx share of revenue:", round(df[df.Prescription_Required == "Yes"][REV].sum() / df[REV].sum(), 3))

**ملاحظات EDA:** الإيراد الشهري ثابت (CV ≈ 5%)، الفروع متقاربة جدًا، طريقة الدفع والنوع مفيهمش فرق يُذكر.
أكبر فروق موجودة بين **فئات الدواء** (Cardiac و Antibiotic و Vitamin الأعلى) — ده اللي هيظهر في أهمية الـ features.

## 3) Feature Engineering & Modeling
نجمّع الداتا على مستوى **فرع × فئة × شهر**، ونبني lags/rolling (مزاحة بشهر عشان منستخدمش المستقبل).
التقسيم زمني: التدريب لحد 2023-03 والاختبار 2023-04..06.

In [ ]:
df["Period"] = df.Transaction_Date.dt.to_period("M")
g = (df.groupby(["Branch_Name", "Medicine_Category", "Period"], observed=True)
       .agg(Revenue=(REV, "sum"), Units=("Quantity", "sum"), Txns=("Transaction_ID", "count")).reset_index())
periods = pd.period_range(df.Period.min(), df.Period.max(), freq="M")
full = pd.MultiIndex.from_product([df.Branch_Name.cat.categories, df.Medicine_Category.cat.categories, periods],
                                  names=["Branch_Name", "Medicine_Category", "Period"])
g = g.set_index(["Branch_Name", "Medicine_Category", "Period"]).reindex(full).fillna(0).reset_index()
g["Branch_Name"] = g.Branch_Name.astype(str); g["Medicine_Category"] = g.Medicine_Category.astype(str)
g = g.sort_values(["Branch_Name", "Medicine_Category", "Period"])
print("Panel:", g.shape)

key = ["Branch_Name", "Medicine_Category"]
grp = g.groupby(key)["Revenue"]
for k in (1, 2, 3): g[f"Rev_lag{k}"] = grp.shift(k)
g["Rev_roll3_mean"] = grp.transform(lambda s: s.shift(1).rolling(3).mean())
g["Rev_roll3_std"] = grp.transform(lambda s: s.shift(1).rolling(3).std())
g["Units_lag1"] = g.groupby(key)["Units"].shift(1)
g["Txns_lag1"] = g.groupby(key)["Txns"].shift(1)
g["Rev_trend"] = g.Rev_lag1 - g.Rev_lag2
g["Month"] = g.Period.dt.month
g["Month_sin"] = np.sin(2 * np.pi * g.Month / 12); g["Month_cos"] = np.cos(2 * np.pi * g.Month / 12)
g["Period_str"] = g.Period.astype(str)
data = g.dropna(subset=["Rev_lag3", "Rev_roll3_mean", "Rev_roll3_std"]).copy()

NUM = ["Rev_lag1","Rev_lag2","Rev_lag3","Rev_roll3_mean","Rev_roll3_std","Rev_trend","Units_lag1","Txns_lag1","Month_sin","Month_cos"]
CAT = ["Branch_Name", "Medicine_Category"]; TARGET = "Revenue"
train = data[data.Period_str <= "2023-03"]; test = data[data.Period_str > "2023-03"]
print(f"train {len(train)} rows | test {len(test)} rows")
data.head()

In [ ]:
def pre(scale):
    return ColumnTransformer([("num", StandardScaler() if scale else "passthrough", NUM),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
models = {
    "Ridge": Pipeline([("pre", pre(True)), ("m", Ridge(alpha=10))]),
    "RandomForest": Pipeline([("pre", pre(False)), ("m", RandomForestRegressor(
        n_estimators=300, min_samples_leaf=5, max_features=0.5, random_state=42, n_jobs=-1))]),
    "GradientBoosting": Pipeline([("pre", pre(False)), ("m", GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=42))]),
}
def metrics(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": np.sqrt(mean_squared_error(y, p)),
            "R2": r2_score(y, p), "WAPE_%": np.abs(y - p).sum() / np.abs(y).sum() * 100}
results = {"Baseline: last month": metrics(test[TARGET], test.Rev_lag1),
           "Baseline: 3-month mean": metrics(test[TARGET], test.Rev_roll3_mean)}

## 4) Model Evaluation

In [ ]:
# Time-series CV (expanding window) على آخر 4 شهور من التدريب
cv_folds = sorted(train.Period_str.unique())[-4:]
for name, pipe in models.items():
    maes = []
    for mth in cv_folds:
        tr, va = train[train.Period_str < mth], train[train.Period_str == mth]
        pipe.fit(tr[NUM + CAT], tr[TARGET]); maes.append(mean_absolute_error(va[TARGET], pipe.predict(va[NUM + CAT])))
    print(f"{name:18s} CV MAE {np.mean(maes):,.0f} +- {np.std(maes):,.0f}")

In [ ]:
preds = {}
for name, pipe in models.items():
    pipe.fit(train[NUM + CAT], train[TARGET])
    preds[name] = pipe.predict(test[NUM + CAT]); results[name] = metrics(test[TARGET], preds[name])
res = pd.DataFrame(results).T.round(3)
best = res.drop(index=[i for i in res.index if i.startswith("Baseline")])["MAE"].idxmin()
print("Best model:", best)
res

In [ ]:
t = test[["Branch_Name", "Medicine_Category", "Period_str", TARGET]].copy()
t["pred"] = preds[best]; t["err"] = t[TARGET] - t.pred

fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
ax[0].scatter(t[TARGET] / 1e3, t.pred / 1e3, alpha=.5, s=14)
lim = [0, t[TARGET].max() / 1e3 * 1.05]; ax[0].plot(lim, lim, "r--")
ax[0].set_xlabel("Actual (k LKR)"); ax[0].set_ylabel("Predicted (k LKR)"); ax[0].set_title(f"Actual vs predicted - {best}")
sns.histplot(t.err / 1e3, bins=40, ax=ax[1]); ax[1].set_title("Residuals (k LKR)")
res["MAE"].sort_values().plot.barh(ax=ax[2], color="tab:green"); ax[2].set_title("Test MAE (lower = better)")
plt.tight_layout(); plt.show()

for col in ["Medicine_Category", "Branch_Name"]:
    e = t.groupby(col).apply(lambda d: d.err.abs().sum() / d[TARGET].sum() * 100).sort_values()
    e.plot.barh(figsize=(8, 5)); plt.title(f"WAPE % by {col}"); plt.tight_layout(); plt.show()

In [ ]:
# Permutation importance (النموذج النهائي متدرب على كل الداتا، فالنتيجة تقريبية)
final = models[best]; final.fit(data[NUM + CAT], data[TARGET])
pi = permutation_importance(final, test[NUM + CAT], test[TARGET], scoring="neg_mean_absolute_error", n_repeats=15, random_state=0)
imp = pd.Series(pi.importances_mean, index=NUM + CAT).sort_values()
imp.plot.barh(figsize=(8, 5)); plt.title("Permutation importance (increase in MAE)"); plt.tight_layout(); plt.show()

**قراءة النتائج:** النموذج أحسن من الـ baseline بفرق بسيط (حوالي 4% في الـ MAE).
الداتا شبه عشوائية حول متوسط كل فئة فمفيش اتجاه أو موسمية قوية يتعلمها الموديل، وأهم feature هي `Medicine_Category`.
الفئات الصغيرة (زي Wellness) أخطاؤها النسبية أعلى.

## 5) Deployment

In [ ]:
# حفظ النموذج + جدول التاريخ اللي هيستخدمه التطبيق
joblib.dump({"model": final, "num": NUM, "cat": CAT, "name": best}, "../models/model.joblib")
g[["Branch_Name", "Medicine_Category", "Period_str", "Revenue", "Units", "Txns"]].to_csv("../data/processed/monthly_history.csv", index=False)
print("saved model.joblib and monthly_history.csv")

In [ ]:
# دالة التوقع (نفس منطق app.py): نبني الـ features للشهر الجاي من آخر 3 شهور
HIST = g[["Branch_Name", "Medicine_Category", "Period_str", "Revenue", "Units", "Txns"]]
NEXT = str(pd.Period(HIST.Period_str.max(), freq="M") + 1)

def forecast_next(branch, category):
    h = HIST[(HIST.Branch_Name == branch) & (HIST.Medicine_Category == category)].sort_values("Period_str").tail(3)
    rev = h.Revenue.to_numpy()[::-1]; month = pd.Period(NEXT, freq="M").month
    row = {"Branch_Name": branch, "Medicine_Category": category, "Rev_lag1": rev[0], "Rev_lag2": rev[1], "Rev_lag3": rev[2],
           "Rev_roll3_mean": rev.mean(), "Rev_roll3_std": rev.std(ddof=1), "Rev_trend": rev[0] - rev[1],
           "Units_lag1": h.Units.iloc[-1], "Txns_lag1": h.Txns.iloc[-1],
           "Month_sin": np.sin(2 * np.pi * month / 12), "Month_cos": np.cos(2 * np.pi * month / 12)}
    return max(float(final.predict(pd.DataFrame([row])[NUM + CAT])[0]), 0)

print(NEXT, "->", round(forecast_next("Pharmacy - Galle", "Cardiac")), "LKR")

### تشغيل كتطبيق ويب
ملف `app/app.py` (Flask) بيقرأ `models/model.joblib` و `data/processed/monthly_history.csv`:

```bash
pip install -r requirements.txt
python3 -m app.app      # من جذر المشروع، http://localhost:5000
```
- `POST /predict` بـ `{"branch": "Pharmacy - Galle", "category": "Cardiac"}`
- `GET /forecast_all` توقع كل الفروع والفئات